## Model Configuration

In [ ]:
import boto3
import os
from langchain_aws import ChatBedrock
from dotenv import load_dotenv
load_dotenv()

# 1. Create a Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=os.getenv("AWS_DEFAULT_REGION", "ap-south-1")
)

# Set up Claude 3 Haiku
llm = ChatBedrock(
    client=bedrock_runtime,
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    model_kwargs={"temperature": 0.0}  # Low temperature for precise extraction
)

## Dummy Data Creation

In [ ]:
import os

# Create data directory structure
os.makedirs("data/docs", exist_ok=True)
os.makedirs("data/faqs", exist_ok=True)
os.makedirs("data/chats", exist_ok=True)

# 1. Realistic Product Documentation
docs_content = """# Microservice Architecture: Payment Gateway Integration
## Overview
The Payment Gateway Microservice handles all third-party transaction processing via Stripe and PayPal. It operates on port 8085.

## Error Handling & Codes
- **ERR_PAY_401**: Unauthorized access token. Occurs if the `X-Gateway-Token` is missing or expired.
- **ERR_PAY_504**: Gateway timeout. The internal timeout limit for connecting to Stripe API is set to 15000ms (15 seconds).

## Database Configuration
The microservice utilizes a Redis cache for transaction state management before committing to PostgreSQL. The cache expiration is configured via `REDIS_TTL_SECONDS`, defaulting to 300 seconds.
"""

with open("data/docs/payment_gateway.md", "w") as f:
    f.write(docs_content)

# 2. Realistic FAQ
faq_content = """# DevOps & Infrastructure FAQ
## Q: How do I restart the staging environment cache?
A: SSH into the staging box and execute `docker restart payment-redis-cache`.

## Q: What should I do if I see an ERR_PAY_504 timeout error in production logs?
A: Check the Stripe Status dashboard first. If Stripe is operational, increase the `GATEWAY_TIMEOUT_MS` environment variable in the deployment chart to `20000` (20 seconds) and trigger a rolling restart.
"""

with open("data/faqs/devops_faq.md", "w") as f:
    f.write(faq_content)

# 3. Noisy, Long Chat History (Simulated Slack Export)
chat_content = """[2026-07-15 09:15:22] dev_dan: Morning team! Quick question, is anyone else getting an ERR_PAY_504 on the staging environment when calling the checkout endpoint?
[2026-07-15 09:16:05] qa_leah: Yeah Dan, I noticed that too during the automated regression run last night. It failed 4 times out of 10.
[2026-07-15 09:18:40] dev_dan: Interesting. Let me check the logs. Looks like it's timing out downstream.
[2026-07-15 09:20:11] lead_sam: Hey folks, regarding the 504s—remember the Redis cache holds transaction states for 300 seconds. If Redis hangs, the gateway times out waiting for the state commit before hitting Postgres.
[2026-07-15 09:22:15] dev_dan: Ah, that makes sense! I checked the container and the staging Redis cache instance was pinned at 100% CPU. I restarted the docker container using the standard `docker restart payment-redis-cache` command and the 504 errors stopped completely.
[2026-07-15 09:23:00] lead_sam: Awesome job troubleshooting, Dan. We should make sure that's captured in the FAQ.
[2026-07-15 11:45:10] dev_dan: Lunch anyone? 🌮
[2026-07-15 11:46:00] qa_leah: Down for tacos!
"""

with open("data/chats/slack_export_2026_07_15.txt", "w") as f:
    f.write(chat_content)

print("📁 Production-grade dummy data generated successfully in the 'data/' folder!")

## Ingestion pipeline

In [1]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_aws import ChatBedrock
import boto3

from dotenv import load_dotenv
load_dotenv()

# --- 1. System Initialization ---
bedrock_client = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
llm = ChatBedrock(
    client=bedrock_client,
    model_id="amazon.nova-lite-v1:0",
    model_kwargs={"temperature": 0.0}
)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Initialize an empty local Chroma database
vectorstore = Chroma(embedding_function=embeddings)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# --- 2. Modular Ingestion Methods ---

def ingest_documentation(file_path):
    """Loads markdown documentation, chunks it simply, and adds to vector store."""
    if not os.path.exists(file_path):
        return f"File {file_path} not found."
    with open(file_path, "r") as f:
        text = f.read()
    chunks = text_splitter.create_documents([text], metadatas=[{"source_type": "documentation", "file": file_path}])
    vectorstore.add_documents(chunks)
    return f"Successfully ingested {len(chunks)} documentation chunks."

def ingest_faqs(file_path):
    """Loads FAQ markdown, chunks it simply, and adds to vector store."""
    if not os.path.exists(file_path):
        return f"File {file_path} not found."
    with open(file_path, "r") as f:
        text = f.read()
    chunks = text_splitter.create_documents([text], metadatas=[{"source_type": "faq", "file": file_path}])
    vectorstore.add_documents(chunks)
    return f"Successfully ingested {len(chunks)} FAQ chunks."

def ingest_chats(file_path):
    """Loads full chat logs, curates Q&As via a single LLM call, and adds to vector store."""
    if not os.path.exists(file_path):
        return f"File {file_path} not found."
    with open(file_path, "r") as f:
        full_chat_text = f.read()
    
    chat_prompt = f"""Extract all technical questions and their verified solutions from this chat log.
    Ignore casual talk.
    
    Chat Log:
    {full_chat_text}
    
    Format:
    Question: [Summary]
    Verified Answer: [Summary]"""
    
    response = llm.invoke(chat_prompt)
    chat_doc = Document(page_content=response.content.strip(), metadata={"source_type": "chat_history", "file": file_path})
    vectorstore.add_documents([chat_doc])
    return "Successfully curated and ingested chat history."

# --- 3. Execute Ingestion ---
# print(ingest_documentation("data/docs/payment_gateway.md"))
# print(ingest_faqs("data/faqs/devops_faq.md"))
print(ingest_chats("data/chats/slack_export_2026_07_15.txt"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully curated and ingested chat history.


## Session History

In [ ]:
# Global dictionary to store histories: { session_id: [messages] }
session_history = {}

def get_session_history(session_id: str) -> list:
    """Retrieves or initializes the chat history for a specific session."""
    if session_id not in session_history:
        session_history[session_id] = []
    return session_history[session_id]

## Query


In [ ]:
def contextualize_query(session_id: str, new_question: str, llm) -> str:
    """Uses chat history to rewrite the user's question into a standalone search query."""
    history = get_session_history(session_id)
    
    # If there is no history, the question is already standalone
    if not history:
        return new_question
        
    # Format the history list into a readable string
    history_text = ""
    for msg in history:
        history_text += f"{msg['role'].capitalize()}: {msg['content']}\n"
        
    # Prompt Claude to rewrite the query based on context
    rewrite_prompt = f"""Given the following conversation history and a new user question, 
    rewrite the new question into a standalone search query that can be used to search a technical vector database.
    Do not answer the question, just reformulate it. If it doesn't need reformulating, return it exactly as is.
    
    Chat History:
    {history_text}
    
    New Question: {new_question}
    
    Standalone Query:"""
    
    response = llm.invoke(rewrite_prompt)
    return response.content.strip()

# Quick Test
session_history["user_123"] = [
    {"role": "user", "content": "Why am I getting a 504 error on the staging environment?"},
    {"role": "assistant", "content": "That usually happens when the payment Redis cache hangs and causes a timeout."}
]

standalone = contextualize_query("user_123", "How do I restart it?", llm)
print(f"Original: How do I restart it?\nRewritten: {standalone}")

## Chat with bot

In [ ]:
# Convert the vector database into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def chat_with_bot(session_id: str, user_question: str, llm, retriever):
    # 1. Get the standalone query for the database
    standalone_query = contextualize_query(session_id, user_question, llm)
    print(f"🔍 Searching vector database for: '{standalone_query}'")
    
    # 2. Retrieve relevant chunks from ChromaDB
    retrieved_docs = retriever.invoke(standalone_query)
    
    # Combine the document text into a single string
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # 3. Format the chat history for the prompt
    history = get_session_history(session_id)
    history_text = "\n".join([f"{msg['role'].capitalize()}: {msg['content']}" for msg in history])
    
    # 4. Construct the Final Prompt 🧩
    final_prompt = f"""You are a helpful engineering assistant. 
    Use the following pieces of retrieved technical context and the conversation history to answer the user's question.
    If you don't know the answer based on the context, just say that you don't know, don't try to make up an answer.
    
    Conversation History:
    {history_text}
    
    Retrieved Context:
    {context_text}
    
    User Question: {user_question}
    
    Answer:"""
    
    # 5. Generate the answer using Claude
    response = llm.invoke(final_prompt)
    bot_answer = response.content.strip()
    
    # 6. Update the in-memory session history
    history.append({"role": "user", "content": user_question})
    history.append({"role": "assistant", "content": bot_answer})
    
    return bot_answer

# --- Let's test the stateful memory! ---
print("--- Turn 1 ---")
ans1 = chat_with_bot("dev_user_1", "What is the default timeout for the API gateway?", llm, retriever)
print(f"🤖 Bot: {ans1}\n")

print("--- Turn 2 ---")
ans2 = chat_with_bot("dev_user_1", "And how do I fix the 504 error related to it?", llm, retriever)
print(f"🤖 Bot: {ans2}\n")

In [9]:
import pandas as pd

In [10]:
df= pd.read_csv("/Users/abdullahasad/development/tech-assist/technical-assistant/data/eval/rag_eval_results.csv")

In [11]:
import openpyxl

In [12]:
df.to_excel("/Users/abdullahasad/development/tech-assist/technical-assistant/data/eval/rag_eval_results.xlsx",index=False)